# CV 4대 태스크 실습 — ③분할 실습2: GrabCut 전경 분할

### 셀 1 — 패키지 설치 및 한글 폰트 설정

In [ ]:
!pip install -q opencv-python-headless matplotlib numpy scikit-image
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

import abc
import os
from dataclasses import dataclass

import cv2
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np

nanum_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if os.path.exists(nanum_path):
    fm.fontManager.addfont(nanum_path)
    plt.rcParams["font.family"] = fm.FontProperties(fname=nanum_path).get_name()
plt.rcParams["axes.unicode_minus"] = False

# ============================================================

### 셀 2 — ===== [도메인 계층 / Domain Layer] =====

In [ ]:
# "전경/배경 분할이란 무엇인가"만 정의하는 순수 타입 계층. cv2에 의존하지 않는다.
# ============================================================


@dataclass
class SegmentationResult:
    """전경/배경 분할 1회 실행의 결과를 표현하는 값 객체(Value Object).

    - foreground_mask: 전경으로 분류된 픽셀을 1, 배경을 0으로 담은 2차원 배열.
    - foreground_ratio: 전체 픽셀 중 전경으로 분류된 비율(0.0~1.0).
    - init_rect: 분할 초기화에 사용한 사각형 힌트(x, y, width, height).
    """

    foreground_mask: np.ndarray
    foreground_ratio: float
    init_rect: tuple


class ImageSegmenter(abc.ABC):
    """'이미지를 전경/배경으로 분할한다'는 핵심 행위만 정의하는 작은 인터페이스(ISP).

    시각화(윤곽선 표시, 배경 블러 등)는 포함하지 않는다(SRP). GrabCut 대신 다른
    전경 분할 알고리즘(예: 딥러닝 기반 세그멘테이션)으로 교체하더라도 이 인터페이스만
    구현하면 되므로 유스케이스 계층은 전혀 수정할 필요가 없다(OCP/LSP).
    """

    @abc.abstractmethod
    def segment(self, image: np.ndarray) -> SegmentationResult:
        """입력 이미지(BGR)를 전경/배경으로 분할하여 SegmentationResult를 반환한다."""
        raise NotImplementedError


# ============================================================

### 셀 3 — ===== [유스케이스 계층 / Use Case Layer] =====

In [ ]:
# 오직 ImageSegmenter 추상화에만 의존하며(DIP), cv2를 직접 호출하지 않는다.
# ============================================================


class SegmentationUseCase:
    """전경/배경 분할 유스케이스.

    생성자 주입으로 ImageSegmenter 구현체를 받아, 어떤 구체 알고리즘이 쓰이는지
    알지 못한 채 "이미지를 분할해 결과를 돌려준다"는 흐름만 책임진다(SRP).
    """

    def __init__(self, segmenter: ImageSegmenter):
        self._segmenter = segmenter

    def run(self, image: np.ndarray) -> SegmentationResult:
        """주입된 분할기로 이미지를 분할해 결과를 반환한다."""
        return self._segmenter.segment(image)


# ============================================================

### 셀 4 — ===== [인프라/어댑터 계층 / Infrastructure Layer] =====

In [ ]:
# cv2 호출은 전부 이 계층에만 존재한다.
# GrabCutSegmenter: 분할 "계산"만 담당 (SRP)
# SegmentationPresenter: 결과를 적용한 파생 이미지(배경 제거/블러/윤곽선)를
#   만드는 "표현" 책임만 담당 — 계산 로직과 분리된다(SRP).
# ============================================================


class GrabCutSegmenter(ImageSegmenter):
    """GrabCut 알고리즘으로 사각형 힌트 하나만으로 전경/배경을 분리하는 어댑터.

    ImageSegmenter 인터페이스의 구체 구현체(Infrastructure) — Domain/UseCase는
    이 클래스의 존재를 모른 채 ImageSegmenter 타입으로만 다룬다(DIP).
    """

    def __init__(self, margin_x_ratio: float = 0.06, margin_y_ratio: float = 0.04, iter_count: int = 8):
        self._margin_x_ratio = margin_x_ratio
        self._margin_y_ratio = margin_y_ratio
        self._iter_count = iter_count

    def segment(self, image: np.ndarray) -> SegmentationResult:
        img_bgr = image

        # GrabCut 초기화 — 전경이 있을 것으로 "대략" 예상되는 사각형(rect)만 지정
        # Watershed(실습1)는 마커를 촘촘히 계산했다면, GrabCut은 대략적인 사각형 힌트 하나로
        # 반복적인 그래프 컷(graph-cut) 최적화를 통해 정교한 전경 마스크를 스스로 찾아낸다.
        h, w = img_bgr.shape[:2]
        margin_x, margin_y = int(w * self._margin_x_ratio), int(h * self._margin_y_ratio)
        rect = (margin_x, margin_y, w - 2 * margin_x, h - 2 * margin_y)  # (x, y, width, height)

        mask = np.zeros((h, w), np.uint8)
        bgd_model = np.zeros((1, 65), np.float64)  # GrabCut 내부용 배경 가우시안 혼합모델 버퍼
        fgd_model = np.zeros((1, 65), np.float64)  # GrabCut 내부용 전경 가우시안 혼합모델 버퍼

        # GrabCut 실행 (반복적으로 전경/배경 확률 모델을 갱신하며 수렴)
        cv2.grabCut(img_bgr, mask, rect, bgd_model, fgd_model, iterCount=self._iter_count, mode=cv2.GC_INIT_WITH_RECT)

        # GrabCut 결과 마스크 값: 0=확실한 배경, 1=확실한 전경, 2=아마도 배경, 3=아마도 전경
        # 1과 3(전경으로 판단된 모든 픽셀)을 최종 전경으로 합친다
        final_mask = np.where((mask == 1) | (mask == 3), 1, 0).astype("uint8")
        fg_ratio = final_mask.sum() / final_mask.size
        print(f"전경으로 분류된 픽셀 비율: {fg_ratio * 100:.1f}%")

        return SegmentationResult(
            foreground_mask=final_mask,
            foreground_ratio=float(fg_ratio),
            init_rect=rect,
        )


class SegmentationPresenter:
    """SegmentationResult를 사람이 보기 좋은 파생 이미지들로 바꾸는 표현 전담 클래스.

    분할 "계산"(GrabCutSegmenter)과는 완전히 분리된 책임이다(SRP) — 배경 제거,
    인물사진 모드(배경 블러), 윤곽선 오버레이 등 "보여주는 방식"이 바뀌어도
    분할 알고리즘/유스케이스에는 영향을 주지 않는다.
    """

    def remove_background(self, image: np.ndarray, result: SegmentationResult) -> np.ndarray:
        """픽셀 단위 마스크로 배경을 제거(검은색 처리)한 이미지를 반환한다."""
        return image * result.foreground_mask[:, :, np.newaxis]

    def portrait_mode(self, image: np.ndarray, result: SegmentationResult, blur_ksize: tuple = (41, 41)) -> np.ndarray:
        """전경은 그대로 두고 배경만 블러 처리한 '인물사진 모드' 이미지를 반환한다."""
        blurred_bg = cv2.GaussianBlur(image, blur_ksize, 0)
        mask = result.foreground_mask
        return np.where(mask[:, :, None] == 1, image, blurred_bg)

    def boundary_overlay(self, image: np.ndarray, result: SegmentationResult) -> np.ndarray:
        """전경 경계선(주황색)과 초기 사각형 힌트(흰색)를 원본 위에 그린 이미지를 반환한다."""
        contours, _ = cv2.findContours(result.foreground_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        overlay = image.copy()
        cv2.drawContours(overlay, contours, -1, (0, 140, 255), 3)
        rect = result.init_rect
        cv2.rectangle(overlay, (rect[0], rect[1]), (rect[0] + rect[2], rect[1] + rect[3]), (255, 255, 255), 1)
        return overlay



### 셀 5 — 샘플 이미지 준비 — 배경이 있는 인물/동물 사진 (전경-배경 분리 실습용)

In [ ]:
# ===== [구성 루트 / Composition Root — Colab 실행 코드] =====
# 여기서부터는 구체 어댑터를 생성하고 유스케이스에 주입한 뒤 실행 → 시각화 → 저장까지
# 수행하는 "조립" 코드만 둔다. 파일 I/O·matplotlib 호출이 허용되는 유일한 계층이다.
from skimage import data as skdata

img_rgb = skdata.chelsea()  # 고양이 사진 (배경: 담요/방바닥)
img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
cv2.imwrite("/content/cat_for_grabcut.jpg", img_bgr)
print("이미지 크기:", img_bgr.shape)


### 셀 6 — 의존성 조립(Composition) — 구체 어댑터 생성 → 유스케이스에 주입

In [ ]:
segmenter: ImageSegmenter = GrabCutSegmenter()             # 인프라 계층의 구체 구현체
segmentation_usecase = SegmentationUseCase(segmenter)       # 유스케이스는 추상 타입만 알고 있음
presenter = SegmentationPresenter()                          # 표현 전담 클래스 (분할기와 분리)


### 셀 7 — 유스케이스 실행 + 결과 적용 — 픽셀 단위 마스크로 배경을 제거(또는 흐림 처리)

In [ ]:
result = segmentation_usecase.run(img_bgr)
final_mask = result.foreground_mask
fg_ratio = result.foreground_ratio
rect = result.init_rect

foreground_only = presenter.remove_background(img_bgr, result)
portrait_mode_img = presenter.portrait_mode(img_bgr, result)
overlay = presenter.boundary_overlay(img_bgr, result)


### 셀 8 — 결과 시각화 (원본+초기 사각형 / 픽셀 마스크 윤곽 / 배경 제거 / 인물사진 모드)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
axes[0].set_title("초기 사각형(흰선) + 분할 경계(주황선)")
axes[1].imshow(final_mask, cmap="gray")
axes[1].set_title(f"픽셀 단위 전경 마스크 ({fg_ratio*100:.0f}%)")
axes[2].imshow(cv2.cvtColor(foreground_only, cv2.COLOR_BGR2RGB))
axes[2].set_title("배경 제거 결과")
axes[3].imshow(cv2.cvtColor(portrait_mode_img, cv2.COLOR_BGR2RGB))
axes[3].set_title("배경 블러(인물사진 모드)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.savefig("/content/grabcut_result.png", dpi=120, bbox_inches="tight")
plt.show()
print("저장 완료: /content/grabcut_result.png")
